# Crop Price Data Cleaning & Merging

This notebook combines crop price data from multiple sources (FAO, World Bank, and WFP) into a single unified dataset. It standardizes column names, crop naming conventions, and data formats across all sources before merging them together and exporting to Stata format.

## Setup and Configuration

Initialize required libraries and set up file paths for reading the processed crop price data from different sources and defining the output location for the merged dataset.


In [73]:
import pandas as pd
import numpy as np
from pathlib import Path

# Base path. Change this to your dropbox folder
mypc_path = "/Users/justinblack/Library/CloudStorage/Dropbox"

# Data and output folder. These should stay the same
data_folder = f'{mypc_path}/Lavoro_Replication/0_data_construction'  
output_folder = f'{data_folder}/1_data_python/temp/'
output_file_name = "combined_crop_prices.csv"
output_path = f"{output_folder}{output_file_name}"
raws_folder = f'{data_folder}/0_data_input/data_processed/crop_prices'

# Read in the raw files
fao_file = pd.read_csv(f"{raws_folder}/fao_crop_prices.csv")
wbk_file = pd.read_csv(f"{raws_folder}/wbk_crop_prices.csv")
wfp_file = pd.read_csv(f"{raws_folder}/wfp_crop_prices.csv")

## Data Processing Functions

Define standardization and processing functions for each data source:

1. Crop name standardization: Clean and normalize crop names across all sources
2. FAO data processing: Transform FAO data from wide to long format, parse dates, and standardize column names
3. World Bank data processing: Melt crop columns, rename fields, and ensure consistent formatting
4. WFP data processing: Parse dates, standardize crop names, and format columns consistently

Each processing function ensures that all datasets have the same column structure: ISO, Month, Year, Crop, Price, Source, Latitude, and Longitude.


In [74]:
# Function to standardize crop names
def standardize_crop_name(crop):
    crop = str(crop).lower()
    # Remove underscores and text in parentheses
    crop = crop.replace('__', '_').replace('_', ' ').strip()
    return crop

country_to_iso = {
    'Guinea': 'GIN',
    'Liberia': 'LBR',
    'Sierra Leone': 'SLE'
}

# Process FAO data
def process_fao(df):
    # Melt the dataframe to get crops in a single column
    id_vars = ['Country', 'Location', 'Month', 'Latitude', 'Longitude']  # Include coordinates in id_vars
    df_melted = df.melt(id_vars=id_vars, var_name='Crop', value_name='Price')
    
    # Parse date. The month is in format "MM/YYYY"
    df_melted['Month'] = pd.to_datetime(df_melted['Month'], format='%m/%Y')    # First, convert to a datetime object
    df_melted['Year'] = df_melted['Month'].dt.year      # Then, extract the year and put it into a new column "Year"
    df_melted['Month'] = df_melted['Month'].dt.month    # Finally, overwrite the Month column with the month number
    
    # Add ISO column
    df_melted['ISO'] = df_melted['Country'].map(country_to_iso)
    
    # Drop country column
    df_melted = df_melted.drop(columns=['Country'])
    
    # Rename Location column
    df_melted = df_melted.rename(columns={
        'Location': 'District',
    })
    
    # Standardize crop names, since FAO has a lot of underscores
    df_melted['Crop'] = df_melted['Crop'].apply(standardize_crop_name)
    
    # Drop rows where Price is NaN
    df_melted = df_melted.dropna(subset=['Price'])
    
    return df_melted[['ISO', 'Month', 'Year', 'Crop', 'Price', 'Latitude', 'Longitude']]

# Process World Bank data
def process_wbk(df):
    crop_columns = ['cassava', 'groundnuts', 'maize', 'millet', 'oil', 'rice', 'sorghum']
    id_vars = ['country', 'lat', 'lon', 'year', 'month', 'mkt_name']
    df_melted = df.melt(id_vars=id_vars, value_vars=crop_columns, var_name='Crop', value_name='Price')
    
    # Add ISO column
    df_melted['ISO'] = df_melted['country'].map(country_to_iso)
    
    # Drop country column
    df_melted = df_melted.drop(columns=['country'])
    
    # Rename columns
    df_melted = df_melted.rename(columns={
        'month': 'Month',
        'year': 'Year',
        'lat': 'Latitude',
        'lon': 'Longitude'
    })
    
    # Drop rows where market name is "Market Average" -- these are market averages, not individual market prices.
    df_melted = df_melted[df_melted['mkt_name'] != 'Market Average']
    
    # Drop rows where Price is NaN
    df_melted = df_melted.dropna(subset=['Price'])
    
    return df_melted[['ISO', 'Month', 'Year', 'Crop', 'Price', 'Latitude', 'Longitude']]

# Process WFP data
def process_wfp(df):
    df = df[df['date'].str.match(r'\d{4}-\d{2}-\d{2}', na=False)].copy()
    # Convert date to Month and Year
    df['date'] = pd.to_datetime(df['date'])
    df['Month'] = df['date'].dt.month
    df['Year'] = df['date'].dt.year
    
    # Rename columns
    df = df.rename(columns={
        'iso': 'ISO',
        'commodity': 'Crop',
        'price': 'Price',
        'latitude': 'Latitude',
        'longitude': 'Longitude'
    })
    
    # Standardize crop names, since WFP has a lot of underscores
    df['Crop'] = df['Crop'].apply(standardize_crop_name)
    
    # Add source column
    df['Source'] = 'WFP'
    
    # Drop rows where Price is NaN
    df = df.dropna(subset=['Price'])
    
    return df[['ISO', 'Month', 'Year', 'Crop', 'Price', 'Latitude', 'Longitude']]

# Process each dataset
fao_processed = process_fao(fao_file)
wbk_processed = process_wbk(wbk_file)
wfp_processed = process_wfp(wfp_file)

# Combine all datasets
combined_df = pd.concat([fao_processed, wbk_processed, wfp_processed], ignore_index=True)

# Convert Price column to numeric, handling any conversion errors
combined_df['Price'] = pd.to_numeric(combined_df['Price'], errors='coerce')

# Drop rows where Price is NaN (after conversion)
combined_df = combined_df.dropna(subset=['Price'])

# Convert other columns to appropriate types for Stata
combined_df['Year'] = combined_df['Year'].astype('int64')
combined_df['Month'] = combined_df['Month'].astype('int64')

# Convert Latitude and Longitude to numeric. 
combined_df['Latitude'] = pd.to_numeric(combined_df['Latitude'], errors='coerce')
combined_df['Longitude'] = pd.to_numeric(combined_df['Longitude'], errors='coerce')

# Sort the final dataset
combined_df = combined_df.sort_values(['ISO', 'Year', 'Month', 'Crop'])

# Drop rows that are duplicates 
combined_df = combined_df.drop_duplicates(subset=['ISO', 'Year', 'Month', 'Crop', 'Price', 'Latitude', 'Longitude'])

# Pivot the crops to columns
# Makes each individual row an ISO-Year-Month-Source-Crop_1_price-Crop_2_price...-Crop_n-Price
combined_df_long = combined_df.pivot(index=['ISO', 'Year', 'Month', 'Latitude', 'Longitude'], columns='Crop', values='Price')

# Make sure each column's first letter is capitalized
combined_df_long.columns = combined_df_long.columns.str[0].str.upper() + combined_df_long.columns.str[1:]

# Reset index to make Latitude and Longitude regular columns
combined_df_long = combined_df_long.reset_index()

# Clean and combine crop column names 
# Beans
# 1. Drop beans  
# 2. Combine beans (fava) with beans (niebe, white) to form one beans column
combined_df_long['Beans'] = combined_df_long['Beans (fava)'].fillna(0) + combined_df_long['Beans (niebe, white)'].fillna(0)
# Replace 0 with NaN where both original columns were NaN
beans_mask = combined_df_long['Beans (fava)'].isna() & combined_df_long['Beans (niebe, white)'].isna()
combined_df_long.loc[beans_mask, 'Beans'] = np.nan
combined_df_long = combined_df_long.drop(columns=['Beans (fava)', 'Beans (niebe, white)'])


# Cane Juice
# 1. Drop cane juice (light)
# 2. Keep cane juice (strong) and rename to cane juice
combined_df_long['Cane juice'] = combined_df_long['Cane juice (strong)']
combined_df_long = combined_df_long.drop(columns=['Cane juice (light)', 'Cane juice (strong)'])

# Cassava
# 1. Create a temporary column to store the final cassava column
combined_df_long['Cassava_Final'] = np.nan

# For SLE, use 'Cassava'
sle_mask = combined_df_long['ISO'] == 'SLE'
combined_df_long.loc[sle_mask, 'Cassava_Final'] = combined_df_long.loc[sle_mask, 'Cassava']

# For GIN, use 'Cassava fresh' (no parentheses) 
gin_mask = combined_df_long['ISO'] == 'GIN'
combined_df_long.loc[gin_mask, 'Cassava_Final'] = combined_df_long.loc[gin_mask, 'Cassava fresh']

# For LBR, use 'Cassava (fresh)' (with parentheses)
lbr_mask = combined_df_long['ISO'] == 'LBR'
combined_df_long.loc[lbr_mask, 'Cassava_Final'] = combined_df_long.loc[lbr_mask, 'Cassava (fresh)']

# Drop the original cassava columns
combined_df_long = combined_df_long.drop(columns=['Cassava', 'Cassava (fresh)', 'Cassava fresh'])

# Rename Cassava_Final to Cassava and convert to numeric
combined_df_long['Cassava_Final'] = pd.to_numeric(combined_df_long['Cassava_Final'], errors='coerce')
combined_df_long = combined_df_long.rename(columns={'Cassava_Final': 'Cassava'})
# Cassava Products
# 1. Drop cassava leaves
# 2. Drop cassava meal (attieke)
# 3. Keep cassava meal (gari) and rename to cassava meal
combined_df_long['Cassava Meal'] = combined_df_long['Cassava meal (gari)']
combined_df_long = combined_df_long.drop(columns=['Cassava leaves', 'Cassava meal (attieke)', 'Cassava meal (gari)'])

# Fish
# 1. Combine fish (bonga) with fish
combined_df_long['Fish'] = combined_df_long['Fish'].fillna(0) + combined_df_long['Fish (bonga)'].fillna(0)
# Replace 0 with NaN where both original columns were NaN
fish_mask = combined_df_long['Fish'].isna() & combined_df_long['Fish (bonga)'].isna()
combined_df_long.loc[fish_mask, 'Fish'] = np.nan
# 2. Drop fish (bonga), fish (dry), fish (herring), fish (smoked), fish bonga
combined_df_long = combined_df_long.drop(columns=['Fish (bonga)', 'Fish (dry)', 'Fish (herring)', 'Fish (smoked)', 'Fish bonga'])
# 3. Turn any 0s to NaN
fish_mask = combined_df_long['Fish'] == 0
combined_df_long.loc[fish_mask, 'Fish'] = np.nan


# Beef
# 1. Drop beef meat (FAO)
# 2. Keep meat (beef) from WFP
combined_df_long['Beef'] = combined_df_long['Meat (beef)']
combined_df_long = combined_df_long.drop(columns=['Beef meat', 'Meat (beef)'])

# Groundnuts
# 1. Drop groundnuts (WBK)
# 2. Keep groundnuts (shelled) from WFP
combined_df_long['Groundnuts'] = combined_df_long['Groundnuts (shelled)']
combined_df_long = combined_df_long.drop(columns=['Groundnuts (shelled)', 'Groundnuts shelled'])

# Maize
# 1. Drop maize flour (FAO)
# 2. Keep maize from WFP
combined_df_long = combined_df_long.drop(columns=['Maize flour'])

# Oil
# 1. Drop palm oil, palm oil traditional
# 2. Keep oil (palm)
combined_df_long = combined_df_long.drop(columns=['Palm oil', 'Palm oil traditional'])

# Onions
# 1. Drop onions (local)
combined_df_long = combined_df_long.drop(columns=['Onions (local)'])

# Peppers
# 1. Combine peppers (fresh) with peppers (green)
combined_df_long['Peppers'] = combined_df_long['Peppers (fresh)'].fillna(0) + combined_df_long['Peppers (green)'].fillna(0)
# Replace 0 with NaN where both original columns were NaN
peppers_mask = combined_df_long['Peppers (fresh)'].isna() & combined_df_long['Peppers (green)'].isna()
combined_df_long.loc[peppers_mask, 'Peppers'] = np.nan
# 2. Drop peppers (fresh), peppers (green)
combined_df_long = combined_df_long.drop(columns=['Peppers (fresh)', 'Peppers (green)'])

# Rice
# 1. Drop rice (FAO) and rice (WBK)
# 2. Drop rice (small grain, imported)
# 3. Drop rice imported, rice parboiled
# 4. Drop rice (white, imported)
# 5. Keep rice (imported) and rice (local) (i.e. NO CHANGE)
combined_df_long = combined_df_long.drop(columns=['Rice', 'Rice (small grain, imported)', 'Rice imported', 'Rice (white, imported)', 'Rice parboiled'])

# Tomatoes
# 1. Drop tomatoes (paste)
combined_df_long = combined_df_long.drop(columns=['Tomatoes (paste)'])

# Other
# 1. Drop exchange rate
# 2. Drop wages 
# 3. Drop fuel 
# 4. Drop other
combined_df_long = combined_df_long.drop(columns=['Exchange rate', 'Wage (non-qualified labour, agricultural)', 'Wage (non-qualified labour, non-agricultural)', 'Fuel (diesel)', 'Fuel (petrol-gasoline)'])

# Make all names stata compatible
combined_df_long.columns = combined_df_long.columns.str.replace(' ', '_')
combined_df_long.columns = combined_df_long.columns.str.replace('(', '')
combined_df_long.columns = combined_df_long.columns.str.replace(')', '')
combined_df_long.columns = combined_df_long.columns.str.replace('/', '_')
combined_df_long.columns = combined_df_long.columns.str.replace('-', '_')
combined_df_long.columns = combined_df_long.columns.str.replace('__', '_')
combined_df_long.columns = combined_df_long.columns.str.replace(',', '_')

# Save the combined dataset
combined_df_long.to_csv(output_path, index=False)
combined_df_long.to_stata(f"{output_folder}test_stata.dta", write_index=False)

print(f"Successfully merged crop prices.")

Successfully merged crop prices.
